In [2]:
from typing import Annotated, TypedDict, List, Dict, Any, Optional
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
from langchain_openai import ChatOpenAI
from langchain_community.agent_toolkits import PlayWrightBrowserToolkit
from langchain_community.tools.playwright.utils import create_async_playwright_browser
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver
from langgraph.prebuilt import ToolNode
from langgraph.graph.message import add_messages
from pydantic import BaseModel, Field
from IPython.display import Image, display
# import gradio as gr
import uuid
import os
from dotenv import load_dotenv

In [3]:
load_dotenv(override=True)
# client = ZaiClient(api_key=os.getenv("ZAI_API_KEY"))

True

In [4]:
import zai

In [ ]:
# from langchain_community.utilities.twilio import TwilioAPIWrapper

# twilio = TwilioAPIWrapper(
#         account_sid="",
#         auth_token="",
#         from_number=""
# )

# # twilio.run("hello world", "")

In [ ]:
# twilio.run("test world", "")

In [7]:
class EvaluatorOutput(BaseModel):
    feedback: str = Field(description="Feedback on assistant's response")
    success_criteria_met: bool = Field(description="Whether the success criteria has been met")
    user_input_needed: bool = Field(description="True if more input is needed from user, or clarifications, or the assistant is stuck")

In [8]:
class State(TypedDict):
    messages: Annotated[List[Any], add_messages]
    success_criteria: str
    feedback_on_work: Optional[str]
    success_criteria_met : bool
    user_input_needed: bool

In [9]:
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_user = os.getenv("PUSHOVER_USER")
pushover_url = "https://api.pushover.net/1/messages.json"

def push(text: str):
    """Send a push notification to user"""
    requests.post(pushover_url, data = {"token": pushover_token, "user": pushover_user, "message": text})


In [10]:
from langchain.agents import create_agent 

In [11]:
glm_llm = ChatOpenAI(
    model="glm-5.1",                                                                                                                                                  
    base_url="https://open.bigmodel.cn/api/paas/v4/",
    api_key=os.getenv("ZAI_API_KEY") # Use ZAI_API_KEY from .env                                                                                                           
) 

agent = create_agent(
    model=glm_llm,
    tools=[push],
    system_prompt="You are a helpful assistant"
)

In [12]:
tools = [agent]

In [13]:
class State(TypedDict):
    messages: Annotated[list, add_messages]

In [14]:
graph_builder= StateGraph(State)

In [16]:
llm = ChatOpenAI(
    model="glm-5.1",
    base_url="https://open.bigmodel.cn/api/paas/v4/",
    api_key=os.getenv("ZAI_API_KEY")
)
llm_with_tools = llm.bind_tools([push])

In [ ]:
def chatbot(state: State):
    return {"messages": [llm_with_tools.invoke(state["messages"])]}

graph_builder.add_node("chatbot", chatbot)


ValueError: Node `chatbot` already present.

In [19]:
graph_builder.add_node("tools", tools=tools)

RuntimeError: 